# Flowgraph: process flows and architecture diagrams

`d3.flowgraph()` draws layered diagrams from an edge list or a Mermaid flowchart. With
`notebook=True` the interactive chart is shown inline (isolated in an iframe): hover a node
to trace its upstream and downstream path, drag nodes, use **▶ Trace**, or open the side
panels for layout, animation and export.

In [ ]:
import pandas as pd
from d3blocks import D3Blocks

# frame=False keeps node_properties as a dict, so nodes can be edited by name below.
d3 = D3Blocks(frame=False, verbose='error')

## 1. Process flow from an edge list

Edges need `source` and `target`; `label` adds text on the edge. Node shapes and swimlanes
(`group`) are set per node.

In [ ]:
edges = pd.DataFrame({
    'source': ['start', 'validate', 'valid', 'valid', 'fix', 'pay', 'ship'],
    'target': ['validate', 'valid', 'pay', 'fix', 'start', 'ship', 'done'],
    'label':  ['', '', 'yes', 'no', 'retry', '', ''],
})
nodes = {
    'start':    {'label': 'Order placed', 'shape': 'terminal', 'group': 'Customer'},
    'validate': {'label': 'Validate cart', 'group': 'Shop'},
    'valid':    {'label': 'Cart valid?', 'shape': 'decision', 'group': 'Shop'},
    'fix':      {'label': 'Show errors', 'group': 'Shop'},
    'pay':      {'label': 'Take payment', 'shape': 'subprocess', 'group': 'Payments'},
    'ship':     {'label': 'Ship parcel', 'shape': 'io', 'group': 'Warehouse'},
    'done':     {'label': 'Delivered', 'shape': 'terminal', 'group': 'Customer'},
}
d3.flowgraph(edges, nodes=nodes, direction='LR', notebook=True, filepath=None, figsize=[None, 520])

Node and edge properties can be edited after drawing; `d3.show()` redraws.
(The trailing `;` stops Jupyter from also printing the returned HTML.)

In [ ]:
d3.node_properties['fix']['color'] = '#e15759'
retry = d3.edge_properties['label'] == 'retry'
d3.edge_properties.loc[retry, ['color', 'style']] = ['#e15759', 'dashed']
d3.show();

## 2. Mermaid flowchart (top-down)

A Mermaid `flowchart` string works as input, and so does Markdown text with a
` ```mermaid ` block or the path to a `.md` / `.mmd` file. `TD` and `TB` both mean top-down.

In [ ]:
runbook = """
flowchart TD
    alert([Alert fires]) --> sev{Customer impact?}
    subgraph triage [Triage]
        sev -->|no| ticket[Open ticket]
        sev -->|yes| declare[[Declare incident]]
    end
    subgraph mitigate [Mitigation]
        declare --> diagnose[Diagnose] --> fixed{Fixed?}
        fixed -->|no| escalate[Page owning team] ==> diagnose
    end
    fixed -->|yes| resolved([Resolved])
    ticket --> resolved
    resolved -.-> review>Postmortem]
    style alert fill:#e15759
    style resolved fill:#59a14f
"""
d3.flowgraph(runbook, notebook=True, filepath=None, figsize=[None, 700])

## 3. Architecture diagram with ELK, edge variants and animation

Groups nest with `parent`. Edge `variant`s set colour and style (`emphasis`, `security`,
`async`, `error`), and `animation` runs per edge. `layout='elk'` gives orthogonal edges
(it adds about 1.5 MB to the output).

In [ ]:
nodes = {
    'users':  {'label': 'Users', 'shape': 'circle'},
    'lb':     {'label': 'Load balancer', 'group': 'Public'},
    'api':    {'label': 'API', 'group': 'App'},
    'auth':   {'label': 'Auth', 'group': 'App'},
    'worker': {'label': 'Worker', 'group': 'App'},
    'db':     {'label': 'PostgreSQL', 'shape': 'database', 'group': 'Data'},
    'bus':    {'label': 'Event bus', 'shape': 'queue', 'group': 'Data'},
    'pay':    {'label': 'Payment provider', 'shape': 'external'},
}
groups = {'VPC': {}, 'Public': {'parent': 'VPC'}, 'App': {'parent': 'VPC'}, 'Data': {'parent': 'VPC'}}
edges = pd.DataFrame([
    ('users', 'lb', 'HTTPS', 'emphasis', 'comet'),
    ('lb', 'api', '', 'emphasis', 'comet'),
    ('api', 'auth', 'gRPC', 'security', None),
    ('api', 'db', 'SQL', 'default', None),
    ('api', 'bus', 'publish', 'async', 'dots'),
    ('bus', 'worker', 'consume', 'async', 'dots'),
    ('worker', 'pay', 'charge', 'security', None),
], columns=['source', 'target', 'label', 'variant', 'animation'])

d3.flowgraph(edges, nodes=nodes, groups=groups, layout='elk', notebook=True, filepath=None, figsize=[None, 560])

## 4. State diagram with a captioned walkthrough

Mermaid `stateDiagram-v2` works too: `[*]` becomes the start and end dots, transition labels are
kept and notes appear in the state's tooltip. `steps` turns **▶ Trace** into **▶ Walk through**:
each step highlights its nodes and edges, dims the rest and zooms to them.

In [ ]:
lifecycle = """
stateDiagram-v2
    [*] --> pending : Signal created
    pending --> awaiting_approval : Risk checks pass
    pending --> cancelled : Strategy disabled
    awaiting_approval --> approved : Operator approves
    awaiting_approval --> rejected : Operator rejects
    approved --> executed : Order sent to broker
    executed --> [*]
    rejected --> [*]
    cancelled --> [*]

    note right of awaiting_approval
        Agent-generated signals
        enter here directly
    end note
"""
steps = [
    {'title': 'Created', 'caption': 'A strategy run creates a pending signal.',
     'highlight': {'edges': [('__start__', 'pending')]}},
    {'title': 'Review', 'caption': 'Signals that pass the risk checks wait for the operator.',
     'highlight': {'nodes': ['awaiting_approval'], 'edges': [('pending', 'awaiting_approval')]}},
    {'title': 'Execution', 'caption': 'Approved signals are sent to the broker.',
     'highlight': {'edges': [('awaiting_approval', 'approved'), ('approved', 'executed')]}},
]
d3.flowgraph(lifecycle, steps=steps, notebook=True, filepath=None, figsize=[None, 650])

## 5. Save as a standalone HTML file

Without `notebook=True` the chart is written to `filepath`; `showfig=True` opens it in the browser.

In [ ]:
d3.flowgraph(edges, nodes=nodes, groups=groups, layout='elk', filepath='flowgraph_architecture.html', showfig=False)
print('Saved', d3.config['filepath'])